# Topic 04: Group-by and aggregate functions

This notebook combines the course-provided lesson material and its exercise section. The answers follow the original prompts and use the pandas methods demonstrated in the lessons. Data files for this topic are stored in the adjacent `Datasets` folder. The requested conda environment is `CISC7204_BonusAssgn_ENV` (Python 3.13).

In [1]:
from pathlib import Path

DATA_DIR = Path("Datasets")

---

## Course lesson: 04_PandasAggregation_01.ipynb

# Intro to Pandas
by Ryan Orsinger

## Module 4: Aggregating
- Using `.crosstab` to count a frequency count for each category pairing
- Using `.pivot_table` to calculate aggregates of numeric values for each category pairing (same as a spreadsheet pivot table)

In [2]:
# Import pandas
import pandas as pd

# Read in some data
df = pd.read_csv(DATA_DIR / "tips.csv")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


### What is `.crosstab?`
- Crosstab computes a simple cross tabulation of two (or more) factors
- Computes a frequency table of factors
- Example: counting up how many tables ate lunch or dinner for each day?
- Example: counting the number of smoking tables broken out by gender?

In [3]:
# Say we needed to get all the different days
df.day.unique()

<StringArray>
['Sun', 'Sat', 'Thur', 'Fri']
Length: 4, dtype: str

In [4]:
# And all the different times
df.time.unique()

<StringArray>
['Dinner', 'Lunch']
Length: 2, dtype: str

In [5]:
# To count Thursday Lunch, we need this compound indexing operation
df[(df.day == "Thur") & (df.time == "Lunch")].shape[0]

61

In [6]:
# To count Thursday Dinner, we need this compound indexing operation
# Repeat this for each day/time combination...
df[(df.day == "Thur") & (df.time == "Dinner")].shape[0]

1

In [7]:
# For another approach,
# we could run .time.value_counts() on each individual day
# But this would be get tedious, too
# Especially if the possible values are numerous
df[df.day == "Thur"].time.value_counts()

time
Lunch     61
Dinner     1
Name: count, dtype: int64

In [8]:
# Crosstab to the rescue!
# Frequency count of all days by all times
pd.crosstab(index=df.day, columns=df.time)

time,Dinner,Lunch
day,,
Fri,12,7
Sat,87,0
Sun,76,0
Thur,1,61


In [9]:
# Margins=True shows the row/column totals
pd.crosstab(index=df.day, columns=df.time, margins=True)

time,Dinner,Lunch,All
day,,,
Fri,12,7,19
Sat,87,0,87
Sun,76,0,76
Thur,1,61,62
All,176,68,244


In [10]:
# Normalize=True show percentages instead of raw counts
pd.crosstab(index=df.day, columns=df.time, margins=True, normalize=True).round(2)

time,Dinner,Lunch,All
day,,,
Fri,0.05,0.03,0.08
Sat,0.36,0.00,0.36
Sun,0.31,0.00,0.31
Thur,0.00,0.25,0.25
All,0.72,0.28,1.00


In [11]:
# We can also pass lists of series into either index or columns
pd.crosstab(index=df.day, columns=[df.time, df.smoker])

time   Dinner     Lunch    
smoker     No Yes    No Yes
day                        
Fri         3   9     1   6
Sat        45  42     0   0
Sun        57  19     0   0
Thur        1   0    44  17

## Using pivot_tables to aggregate more than counts
- Use `.pivot_table` to set up category pairings, then specify the column to measure, in aggregate, and your aggregate function(s)
- The `.pivot_table` method defaults to using the average, 
- We can specify multiple categories in the index and columns, but the results can become visually busy
- Example: for each day/time pairing, calculate the average `total_bill`
- Example: for each day/time pairing, get the average `total_bill` and `tip`
- Example: for each day/time pairing, calculate the min, median, max `tip`

In [12]:
# Without specifying a "values" column, 
# pivot_table returns the numeric average of numeric columns, broken out by each category pair
pd.pivot_table(df, index="day", columns="time", values=["total_bill", "tip", "size"])

size                 tip           total_bill           
time    Dinner     Lunch    Dinner     Lunch     Dinner      Lunch
day                                                               
Fri   2.166667  2.000000  2.940000  2.382857  19.663333  12.845714
Sat   2.517241       NaN  2.993103       NaN  20.441379        NaN
Sun   2.842105       NaN  3.255132       NaN  21.410000        NaN
Thur  2.000000  2.459016  3.000000  2.767705  18.780000  17.664754

In [13]:
# Use the values argument to specify numeric column(s)
pd.pivot_table(df, index="day", columns="time", values="total_bill")

time,Dinner,Lunch
day,,
Fri,19.663333,12.845714
Sat,20.441379,NaN
Sun,21.410000,NaN
Thur,18.780000,17.664754


In [14]:
# Use the "values" argument to specify which columns to calculate
pd.pivot_table(df, index="day", columns="time", values=["total_bill", "tip"])

tip           total_bill           
time    Dinner     Lunch     Dinner      Lunch
day                                           
Fri   2.940000  2.382857  19.663333  12.845714
Sat   2.993103       NaN  20.441379        NaN
Sun   3.255132       NaN  21.410000        NaN
Thur  3.000000  2.767705  18.780000  17.664754

In [15]:
# Use the aggfunc argument to overwrite the default mean function
pd.pivot_table(df, values="tip", aggfunc="median", index="day", columns="time")

time,Dinner,Lunch
day,,
Fri,3.00,2.2
Sat,2.75,NaN
Sun,3.15,NaN
Thur,3.00,2.3


In [16]:
# The aggfunc argument can take a list of aggregate functions
pd.pivot_table(df, values="tip", aggfunc=["min", "median", "max"], index="day", columns="time")

min       median          max      
time Dinner Lunch Dinner Lunch Dinner Lunch
day                                        
Fri    1.00  1.58   3.00   2.2   4.73  3.48
Sat    1.00   NaN   2.75   NaN  10.00   NaN
Sun    1.01   NaN   3.15   NaN   6.50   NaN
Thur   3.00  1.25   3.00   2.3   3.00  6.70

## Additional Resources
- https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html
- https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html

## Exercises
- Use crosstab on the `tips` dataframe to count the number of differently sized tables for each time of day. *Hint* remember that `.size` is a built-in attribute on pandas objects.
- Use `pd.read_csv` and the `mpg.csv` file to create a dataframe named `mpg`.
- Use `.crosstab` to count the number of vehicles for each combination of class and drivetrain. *Hint* remember that `class` is a reserved word in Python.
- Use `.crosstab` to count the number of vehicles for each combination of manufacturer and drivetrain.
- Use `.pivot_table` and `mpg` to calculate the average highway mileage for each combination of vehicle class and drivetrain. 
- Use `.pivot_table` and `mpg` to calculate the median city mileage for each combination of manufacturer and drivetrain.

In [17]:
# Use crosstab on the tips dataframe to count the number of differently sized tables for each time of day. 
# Hint remember that .size is a built-in attribute on pandas objects.


tips = df
pd.crosstab(tips["time"], tips["size"])

size,1,2,3,4,5,6
time,,,,,,
Dinner,2,104,33,32,4,1
Lunch,2,52,5,5,1,3


In [18]:
# Use pd.read_csv and the mpg.csv file to create a dataframe named mpg.


mpg = pd.read_csv(DATA_DIR / "mpg.csv")
mpg.head()

,manufacturer,model,displ,year,cyl,trans,drv,cty,hwy,fl,class
0,audi,a4,1.8,1999,4,auto(l5),f,18,29,p,compact
1,audi,a4,1.8,1999,4,manual(m5),f,21,29,p,compact
2,audi,a4,2.0,2008,4,manual(m6),f,20,31,p,compact
3,audi,a4,2.0,2008,4,auto(av),f,21,30,p,compact
4,audi,a4,2.8,1999,6,auto(l5),f,16,26,p,compact


In [19]:
# Use .crosstab to count the number of vehicles for each combination of class and drivetrain. 
# Hint remember that "class" is a reserved word in Python.


pd.crosstab(mpg["class"], mpg["drv"])

drv,4,f,r
class,,,
2seater,0,0,5
compact,12,35,0
midsize,3,38,0
minivan,0,11,0
pickup,33,0,0
subcompact,4,22,9
suv,51,0,11


In [20]:
# Use .crosstab to count the number of vehicles for each combination of manufacturer and drivetrain.


pd.crosstab(mpg["manufacturer"], mpg["drv"])

drv,4,f,r
manufacturer,,,
audi,11,7,0
chevrolet,4,5,10
dodge,26,11,0
ford,13,0,12
honda,0,9,0
hyundai,0,14,0
jeep,8,0,0
land rover,4,0,0
lincoln,0,0,3


In [21]:
# Use .pivot_table and mpg to calculate the average highway mileage for each combination of vehicle class and drivetrain.


pd.pivot_table(
    mpg,
    values="hwy",
    index="class",
    columns="drv",
    aggfunc="mean"
)

drv,4,f,r
class,,,
2seater,NaN,NaN,24.800000
compact,25.833333,29.142857,NaN
midsize,24.000000,27.552632,NaN
minivan,NaN,22.363636,NaN
pickup,16.878788,NaN,NaN
subcompact,26.000000,30.545455,23.222222
suv,18.274510,NaN,17.454545


In [22]:
# Use .pivot_table and mpg to calculate the median city mileage for each combination of manufacturer and drivetrain.


pd.pivot_table(
    mpg,
    values="cty",
    index="manufacturer",
    columns="drv",
    aggfunc="median"
)

drv,4,f,r
manufacturer,,,
audi,17.0,18.0,NaN
chevrolet,12.5,18.0,14.5
dodge,12.5,16.0,NaN
ford,13.0,NaN,15.0
honda,NaN,24.0,NaN
hyundai,NaN,18.5,NaN
jeep,14.0,NaN,NaN
land rover,11.5,NaN,NaN
lincoln,NaN,NaN,11.0


---

## Course lesson: 04_PandasAggregation_02.ipynb

# Intro to Pandas
by Ryan Orsinger

## Module 4: Aggregating (continued)
- Using `.groupby` and aggregate methods
- Understanding the `.groupby` object
- Introducing the `.agg` method
- Specifying column output
- Grouping by multiple columns

In [23]:
# Import pandas
import pandas as pd

# Read in our data
df = pd.read_csv(DATA_DIR / "tips.csv")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [24]:
# We've already worked with some aggregate functions
df.total_bill.median()

np.float64(17.795)

In [25]:
# Aggregate functions run on entire columns or dataframes
df.mean(numeric_only=True)

total_bill    19.785943
tip            2.998279
size           2.569672
dtype: float64

In [26]:
df.tip.min(), df.tip.max()

(np.float64(1.0), np.float64(10.0))

In [27]:
# .describe is also an aggregate function, since it is host to multiple aggregate functions
df.tip.describe()

count    244.000000
mean       2.998279
std        1.383638
min        1.000000
25%        2.000000
50%        2.900000
75%        3.562500
max       10.000000
Name: tip, dtype: float64

But what do we do when we need aggregate results for each value in a categorical column?

In [28]:
# It's possible to manually create dataframes for each category
# But this can become tedious with many categories
# and with multiple columns
# Especially if we want to run the same methods on each dataframe
# etc...
thurs = df[df.day == "Thur"]
fri = df[df.day == "Fri"]
sat = df[df.day == "Sat"]
sun = df[df.day == "Sun"]

# We don't have labels with this method, unfortunately
thurs.total_bill.mean(), fri.total_bill.mean(), sat.total_bill.mean(), sun.total_bill.mean()

(np.float64(17.682741935483868),
 np.float64(17.15157894736842),
 np.float64(20.441379310344825),
 np.float64(21.41))

In [29]:
# We calculate from the groupby object with aggregate methods (.mean, .median, etc...)
# Calculate the average total bill for each day
# The "for each" means that we're grouping by the day column
df.groupby("day").total_bill.mean()

day
Fri     17.151579
Sat     20.441379
Sun     21.410000
Thur    17.682742
Name: total_bill, dtype: float64

In [30]:
# The groupby object is a compound entity, built for accessing with aggregate functions
df.groupby("day")

In [31]:
# The groupby object does not print out results, 
# Underneath the hood, it is an object containing multiple tuples of dataframes for each possible categorical value
# Recommend avoiding decomposing groupby objects (this cell is to share context)
# That's what aggregate functions are for!
a, b, c, d = df.groupby("day")
a

('Fri',
      total_bill   tip     sex smoker  day    time  size
 90        28.97  3.00    Male    Yes  Fri  Dinner     2
 91        22.49  3.50    Male     No  Fri  Dinner     2
 92         5.75  1.00  Female    Yes  Fri  Dinner     2
 93        16.32  4.30  Female    Yes  Fri  Dinner     2
 94        22.75  3.25  Female     No  Fri  Dinner     2
 95        40.17  4.73    Male    Yes  Fri  Dinner     4
 96        27.28  4.00    Male    Yes  Fri  Dinner     2
 97        12.03  1.50    Male    Yes  Fri  Dinner     2
 98        21.01  3.00    Male    Yes  Fri  Dinner     2
 99        12.46  1.50    Male     No  Fri  Dinner     2
 100       11.35  2.50  Female    Yes  Fri  Dinner     2
 101       15.38  3.00  Female    Yes  Fri  Dinner     2
 220       12.16  2.20    Male    Yes  Fri   Lunch     2
 221       13.42  3.48  Female    Yes  Fri   Lunch     2
 222        8.58  1.92    Male    Yes  Fri   Lunch     1
 223       15.98  3.00  Female     No  Fri   Lunch     3
 224       13.42  1.58 

In [32]:
# We calculate from the groupby object with aggregate methods (.mean, .median, etc...)
# Calculate the average total bill for each day
# The "for each" means that we're grouping by the day column
df.groupby("day").total_bill.mean()

day
Fri     17.151579
Sat     20.441379
Sun     21.410000
Thur    17.682742
Name: total_bill, dtype: float64

In [33]:
# Consider the following
# We get the average for each day, on all numeric columns
# Notice that each groupby result redefines what each row means
df.groupby("day")[["total_bill", "tip", "size"]].mean()

,total_bill,tip,size
day,,,
Fri,17.151579,2.734737,2.105263
Sat,20.441379,2.993103,2.517241
Sun,21.410000,3.255132,2.842105
Thur,17.682742,2.771452,2.451613


In [34]:
# We can also group by more than 1 column. This creates a multiple
# Without specifying the columns, we'll see all the numeric columns in the output
df.groupby(["day", "time"])[["total_bill", "tip", "size"]].mean()

total_bill       tip      size
day  time                                  
Fri  Dinner   19.663333  2.940000  2.166667
     Lunch    12.845714  2.382857  2.000000
Sat  Dinner   20.441379  2.993103  2.517241
Sun  Dinner   21.410000  3.255132  2.842105
Thur Dinner   18.780000  3.000000  2.000000
     Lunch    17.664754  2.767705  2.459016

In [35]:
# We can also group by more than 1 column. This creates a multiple
# We can provide a list of numeric columns inside the square brackets that specify columns (making double brackets)
df.groupby(["day", "time"])[["total_bill", "tip"]].mean()

total_bill       tip
day  time                        
Fri  Dinner   19.663333  2.940000
     Lunch    12.845714  2.382857
Sat  Dinner   20.441379  2.993103
Sun  Dinner   21.410000  3.255132
Thur Dinner   18.780000  3.000000
     Lunch    17.664754  2.767705

In [36]:
# If we need to turn the groupby output into their own column names, we can use .reset_index
df.groupby(["day", "time"])[["total_bill", "tip"]].mean().reset_index()

,day,time,total_bill,tip
0,Fri,Dinner,19.663333,2.940000
1,Fri,Lunch,12.845714,2.382857
2,Sat,Dinner,20.441379,2.993103
3,Sun,Dinner,21.410000,3.255132
4,Thur,Dinner,18.780000,3.000000
5,Thur,Lunch,17.664754,2.767705


In [37]:
df.groupby("day")[["total_bill", "tip"]].mean()

,total_bill,tip
day,,
Fri,17.151579,2.734737
Sat,20.441379,2.993103
Sun,21.410000,3.255132
Thur,17.682742,2.771452


In [38]:
# .describe is an aggregate function, too
df.groupby("time").total_bill.describe()

,count,mean,std,min,25%,50%,75%,max
time,,,,,,,,
Dinner,176.0,20.797159,9.142029,3.07,14.4375,18.390,25.2825,50.81
Lunch,68.0,17.168676,7.713882,7.51,12.2350,15.965,19.5325,43.11


In [39]:
# Using the .agg method to specify multiple
df.groupby("day").total_bill.agg(["mean", "std"])

,mean,std
day,,
Fri,17.151579,8.302660
Sat,20.441379,9.480419
Sun,21.410000,8.832122
Thur,17.682742,7.886170


In [40]:
# Using the .agg method to specify multiple
# We can cal .agg on multiple numeric columns, too
df.groupby("day")[["total_bill", "tip"]].agg(["mean", "std"])

total_bill                 tip          
           mean       std      mean       std
day                                          
Fri   17.151579  8.302660  2.734737  1.019577
Sat   20.441379  9.480419  2.993103  1.631014
Sun   21.410000  8.832122  3.255132  1.234880
Thur  17.682742  7.886170  2.771452  1.240223

In [41]:
# Since the output is a dataframe, we can transpose it, if doing so makes for easier reading
df.groupby("day")[["total_bill", "tip"]].agg(["mean", "std"]).T

day                    Fri        Sat        Sun       Thur
total_bill mean  17.151579  20.441379  21.410000  17.682742
           std    8.302660   9.480419   8.832122   7.886170
tip        mean   2.734737   2.993103   3.255132   2.771452
           std    1.019577   1.631014   1.234880   1.240223

## The forms of .groupby

| specific example    |  general form    |
| ---- | ---- |
|`df.groupby("day").mean()` | `df.groupby("categorical_column").aggregate_function()`     |
| `df.groupby("day").total_bill.mean()`     | `df.groupby("categorical_column").numeric_column.aggregate_function()`     |
| `df.groupby("day")["tip"].median()`     | `df.groupby("categoryA")["numeric_columnA"].aggregate_function()`     |
| `df.groupby("day")[["total_bill", "tip"]].min()`     | `df.groupby("categoryA")[["numeric_columnA", "numeric_columnB"]].aggregate_function()`     |
| `df.groupby(["day", "time"]).mean()`     | `df.groupby(["categoryA", "categoryB").aggregate_function()` |
| `df.groupby("day").agg(["min", "median", "max"])`    | `df.groupby("category").agg(["min", "median", "max"])`     |
| `df.groupby("day")[["total_bill", "tip"]].agg(["min", "median", "max"])`    | `df.groupby("category")[["numericA", "numericB"]].agg(["min", "median", "max"])`     |

## Additional Resource
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html
- Further reading on the multi-index https://pandas.pydata.org/docs/user_guide/advanced.html from grouping by multiple columns

## Exercises
- Use the "mpg.csv" dataset to create a dataframe named `mpg`
- Group by manufacturer and obtain the highest `hwy` mileage for each manufacturer
- Group by the manufacturer and obtain the average `hwy` and `cty` mileage
- Group by the number of cylinders and get the average displacement for each cylinder
- Group by the vehicle class, then calculate the average and standard deviation of `hwy` mileage
- Which vehicle class has the largest standard deviation of hwy mileage?

In [42]:
# Use the DATA_DIR / "mpg.csv" dataset to create a dataframe named `mpg`


mpg = pd.read_csv(DATA_DIR / "mpg.csv")
mpg.head()

,manufacturer,model,displ,year,cyl,trans,drv,cty,hwy,fl,class
0,audi,a4,1.8,1999,4,auto(l5),f,18,29,p,compact
1,audi,a4,1.8,1999,4,manual(m5),f,21,29,p,compact
2,audi,a4,2.0,2008,4,manual(m6),f,20,31,p,compact
3,audi,a4,2.0,2008,4,auto(av),f,21,30,p,compact
4,audi,a4,2.8,1999,6,auto(l5),f,16,26,p,compact


In [43]:
# Group by manufacturer and obtain the highest hwy mileage for each manufacturer


mpg.groupby("manufacturer")["hwy"].max()

manufacturer
audi          31
chevrolet     30
dodge         24
ford          26
honda         36
hyundai       31
jeep          22
land rover    18
lincoln       18
mercury       19
nissan        32
pontiac       28
subaru        27
toyota        37
volkswagen    44
Name: hwy, dtype: int64

In [44]:
# Group by the manufacturer and obtain the average hwy and cty mileage


mpg.groupby("manufacturer")[["hwy", "cty"]].mean()

,hwy,cty
manufacturer,,
audi,26.444444,17.611111
chevrolet,21.894737,15.000000
dodge,17.945946,13.135135
ford,19.360000,14.000000
honda,32.555556,24.444444
hyundai,26.857143,18.642857
jeep,17.625000,13.500000
land rover,16.500000,11.500000
lincoln,17.000000,11.333333


In [45]:
# Group by the number of cylinders and get the average displacement for each cylinder


mpg.groupby("cyl")["displ"].mean()

cyl
4    2.145679
5    2.500000
6    3.408861
8    5.132857
Name: displ, dtype: float64

In [46]:
# Group by the vehicle class, then calculate the average and standard deviation of hwy mileage


mpg.groupby("class")["hwy"].agg(["mean", "std"])

,mean,std
class,,
2seater,24.800000,1.303840
compact,28.297872,3.781620
midsize,27.292683,2.135930
minivan,22.363636,2.062655
pickup,16.878788,2.274280
subcompact,28.142857,5.375012
suv,18.129032,2.977973


In [47]:
# Which vehicle class has the largest standard deviation of hwy mileage?


class_hwy_stats = mpg.groupby("class")["hwy"].agg(["mean", "std"])
class_hwy_stats["std"].idxmax()

'subcompact'